In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp , input_file_name , lit

# Spark is already running in fabric
spark

df_bronze = spark.read.csv("Files/bronze/transactions/transactions_raw.csv", header=True , inferSchema=True)

print(f"Row count: {df_bronze.count()}")
df_bronze.printSchema





StatementMeta(, 2a098115-9e60-4dfa-bdab-1615c02183b9, 3, Finished, Available, Finished, False)

Row count: 10100


<bound method DataFrame.printSchema of DataFrame[transaction_id: string, date: date, merchant: string, category: string, amount: double, currency: string, status: string]>

In [2]:
# Addd the Metadata Columns

df_bronze_enriched =df_bronze \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("sourcefile", input_file_name()) \
    .withColumn("layer", lit("bronze"))

df_bronze_enriched.printSchema    

StatementMeta(, 2a098115-9e60-4dfa-bdab-1615c02183b9, 4, Finished, Available, Finished, False)

<bound method DataFrame.printSchema of DataFrame[transaction_id: string, date: date, merchant: string, category: string, amount: double, currency: string, status: string, ingestion_timestamp: timestamp, sourcefile: string, layer: string]>

In [3]:
# Write as Delta table to the Tables section
df_bronze_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_transactions")

print("Bronze Delta table created successfully")

StatementMeta(, 2a098115-9e60-4dfa-bdab-1615c02183b9, 5, Finished, Available, Finished, False)

Bronze Delta table created successfully


In [10]:
# read back from the delta table to confirm
df_check = spark.read.format("delta").table("bronze_transactions")

print(f"Total rows: {df_check.count()}")
print(f"Total columns: {df_check.columns}")


# check for thr meta data we added
df_check.select(
    "transaction_id",
    "amount",
    "ingestion_timestamp",
    "sourcefile"
).show(5, truncate=False)

StatementMeta(, 2a098115-9e60-4dfa-bdab-1615c02183b9, 12, Finished, Available, Finished, False)

Total rows: 10100
Total columns: ['transaction_id', 'date', 'merchant', 'category', 'amount', 'currency', 'status', 'ingestion_timestamp', 'sourcefile', 'layer']
+--------------+------+--------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|transaction_id|amount|ingestion_timestamp       |sourcefile                                                                                                                                                                                            |
+--------------+------+--------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|TXN00001      |13.42 |2026-03-02 02:46:46.560847|abfss://09a5f7ed-885d-485c-a30f-e77496

In [22]:
# Data Exploration

from pyspark.sql.functions import col , count , isnan, when, sum

print("=== NULL CHECK ===")

df_check.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_check.columns
]).show()

print("=== DUPLICATE CHECK ===")

total = df_check.count()
distinct = df_check.dropDuplicates(["transaction_id"]).count()
print(f"Tota; rows : {total} | Distinct transaction_ids: {distinct} | Duplicates: {total - distinct} ")



print("=== STATUS VALUES ===")
df_check.groupBy("status").count().show()


print("=== AMOUNT RANGE ===")
df_check.select("amount").summary("min", "max", "mean", "stddev").show()

StatementMeta(, 2a098115-9e60-4dfa-bdab-1615c02183b9, 24, Finished, Available, Finished, False)

=== NULL CHECK ===
+--------------+----+--------+--------+------+--------+------+-------------------+----------+-----+
|transaction_id|date|merchant|category|amount|currency|status|ingestion_timestamp|sourcefile|layer|
+--------------+----+--------+--------+------+--------+------+-------------------+----------+-----+
|             0|   0|       0|     100|   201|       0|     0|                  0|         0|    0|
+--------------+----+--------+--------+------+--------+------+-------------------+----------+-----+

=== DUPLICATE CHECK ===
Tota; rows : 10100 | Distinct transaction_ids: 10000 | Duplicates: 100 
=== STATUS VALUES ===
+---------+-----+
|   status|count|
+---------+-----+
|completed| 5950|
|   failed| 2054|
|  pending| 2096|
+---------+-----+

=== AMOUNT RANGE ===
+-------+------------------+
|summary|            amount|
+-------+------------------+
|    min|              3.54|
|    max|            399.97|
|   mean|202.62540862713362|
| stddev|114.22812348776733|
+-------+--